# Chapter 9 — Assembling MAS with Subagents

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

In [9]:
!pip install llm-agents-from-scratch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.6/105.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.1/255.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 21.2 MB/s eta 0:00:00


## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [10]:
import os
from google.colab import userdata

# Get the API key from Colab secrets
ollama_api_key = userdata.get('OLLAMA_API_KEY')

# Set the environment variable
if ollama_api_key:
    os.environ['OLLAMA_API_KEY'] = ollama_api_key
    print("OLLAMA_API_KEY successfully set in the environment.")
else:
    print("OLLAMA_API_KEY not found in Colab secrets. Please add it.")

OLLAMA_API_KEY successfully set in the environment.


In [11]:
import os, shutil, subprocess, time, urllib.request, urllib.error


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    # Lightning persistent path first, then standard locations
    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh"
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
print(use_cloud)
ensure_ollama() if not use_cloud else print("✓ Using Ollama Cloud")

True
✓ Using Ollama Cloud


## Examples

### Example 1: Manual Dispatch with UseSubAgentTool

Console logging is enabled below so you can watch the dispatched sub-agent run — its log lines are tagged `[hailstone]`, distinguishing them from the coordinator's own logs in later examples.

In [12]:
import logging

from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.data_structures import ToolCall
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.subagents import SubAgentSpec, UseSubAgentTool
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)

model = "gpt-oss:120b" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None
llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)

spec = SubAgentSpec(
    name="hailstone",
    description="Computes Hailstone sequences using next_number.",
    builder=LLMAgentBuilder(llm=llm, tools=[next_number_tool]),
    max_steps=20,
)
tool = UseSubAgentTool(subagents_registry={spec.name: spec})

tool_call = ToolCall(
    tool_name="from_scratch__use_subagent",
    arguments={
        "name": "hailstone",
        "task": (
            "Compute the full Hailstone sequence for 5 step by step "
            "using next_number, until you reach 1. Report how many "
            "steps it took."
        ),
    },
)
result = await tool(tool_call=tool_call)
print(result.content)

ValidationError: 1 validation error for SubAgentSpec
agent
  Field required [type=missing, input_value={'name': 'hailstone', 'de...cc6e0>, 'max_steps': 20}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

### Example 2: Automatic Dispatch via the Coordinator

In [ ]:
from llm_agents_from_scratch.data_structures import Task

# reuses `spec` from Example 1 -- the same hailstone SubAgentSpec, no
# changes needed. SubAgentSpec.builder builds a fresh LLMAgent per
# dispatch, so it's safe to reuse across coordinators.
hailstone_coordinator = await (
    LLMAgentBuilder().with_llm(llm).with_subagents([spec]).build()
)

task_hailstone = Task(
    instruction="Determine the Hailstone sequence for the number 5.",
)
result_hailstone = await hailstone_coordinator.run(task_hailstone)
print(result_hailstone.content)

### Example 3: HITL on the Coordinator

In [ ]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.tools.default import SharedConsoleHumanInputTool

# both subagents here do simple, tightly-scoped work, so a small
# local model is plenty
slm = OllamaLLM(model="qwen3:8b", think=False)

# two subagents, each with its own SharedConsoleHumanInputTool instance
# -- but the lock is a CLASS-level asyncio.Lock, so both instances
# share it. Dispatched concurrently, their prompts serialize instead
# of racing for stdin.
hailstone_input_tool = SharedConsoleHumanInputTool(agent_name="hailstone")
hailstone_spec = SubAgentSpec(
    name="hailstone",
    description=(
        "Asks the human operator for a starting number, then computes "
        "its Hailstone sequence."
    ),
    builder=LLMAgentBuilder(
        llm=slm,
        tools=[next_number_tool, hailstone_input_tool],
    ),
    max_steps=20,
)

greeter_input_tool = SharedConsoleHumanInputTool(agent_name="greeter")
greeter_spec = SubAgentSpec(
    name="greeter",
    description="Asks the human operator for their name, then greets them.",
    builder=LLMAgentBuilder(llm=slm, tools=[greeter_input_tool]),
    max_steps=5,
)

# the builder's only async work is MCP tool discovery, and this
# coordinator has no MCP providers, so construct the LLMAgent directly.
# The subagent specs above still use builders: SubAgentSpec.builder is a
# recipe, rebuilt fresh on every dispatch.
hitl_coordinator = LLMAgent(
    llm=llm,
    subagents=[hailstone_spec, greeter_spec],
)

task_hitl = Task(
    instruction=(
        "Dispatch both the hailstone and greeter subagents in the same "
        "response so they run concurrently. hailstone should ask the "
        "human for a starting number, then compute its Hailstone "
        "sequence. greeter should ask the human for their name, then "
        "return a friendly greeting. Report both results."
    ),
)
# watch the console: the two prompts complete one at a time, never
# interleaved, thanks to SharedConsoleHumanInputTool's shared lock.
result_hitl = await hitl_coordinator.run(task_hitl)
print(result_hitl.content)

### Example 4: Router/Triage

In [ ]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.subagents.recipes import explore_subagent, general_subagent

coordinator = LLMAgent(
    llm=llm,  # the bigger model from Example 1
    subagents=[general_subagent(llm), explore_subagent(llm)],
)

# a lookup task — should route to explore (reads hailstone_known_sequences.json)
task_lookup = Task(
    instruction=(
        "Look up the Hailstone sequence for 6 in "
        "hailstone_known_sequences.json and report it."
    ),
)
result_lookup = await coordinator.run(task_lookup)
print(result_lookup.content)

### Example 5: Parallel Fan-Out

In [ ]:
task_fanout = Task(
    instruction=(
        "Dispatch three sub-agent calls in the same response so they run "
        "concurrently: use general to compute the Hailstone sequence for "
        "4, use general to compute it for 8, and use explore to look up "
        "the sequence for 12 in hailstone_known_sequences.json. Report "
        "which of the three starting numbers took the most steps."
    ),
)
# gather() runs the three dispatches concurrently, but wall-clock
# speedup isn't guaranteed: a single local Ollama instance queues
# concurrent requests to the *same* model unless OLLAMA_NUM_PARALLEL is
# raised. Cloud models parallelize for free.
result_fanout = await coordinator.run(task_fanout)
print(result_fanout.content)

### Example 6: Chaining

In [ ]:
coordinator.subagents_registry[spec.name] = spec

chain_task = Task(
    instruction=(
        "Ask explore to read hailstone_known_sequences.json and report "
        "its contents. Compare the recorded step counts yourself and "
        "take the starting number with the most steps. Then ask "
        "hailstone to compute the full Hailstone sequence for double "
        "that number. Report the sequence and whether it took more "
        "steps than the recorded entry."
    ),
)
chain_handler = coordinator.run(chain_task, max_steps=10)
chain_result = await chain_handler
print(chain_result.content)

`chain_handler.rollout` is what shows this really was a chain. `explore` reports what the file contains, then the coordinator picks the entry with the most steps, doubles it, and writes the result into `hailstone`'s dispatch instruction. That second dispatch could not have been issued until the first one answered, and the number in it appears nowhere in the original task. The coordinator does not just relay a value here, it transforms one.

The doubling also gives us a free correctness check. Any even number `2n` steps straight down to `n`, so a doubled starting number always takes exactly one more step than the original.

In [ ]:
print(chain_handler.rollout)